# Browse Disagreement Plots

Lightweight viewer for pre-saved comparison plots from `compare_runs_tables_disagreement_STYLED.ipynb`.

**No model loading** — only reads PNGs from disk.

Folder structure: `comparison_outputs/disagreement_plots/{baseline}_vs_{compare}_{metric}/`

In [1]:
# ----------------------------
# 1) Configure path to plots folder
# ----------------------------
import json
from pathlib import Path

# Option A: Set full path directly
# Format: comparison_outputs/{YYYY-MM-DD_HH-MM-SS}/disagreement_plots/{baseline}_vs_{compare}_{metric}
PLOTS_DIR = Path("./comparison_outputs/2025-02-14_15-30-45/disagreement_plots/SAM-H__New_vs_FiLM_Rosie_Weights_bPQ")

# Option B: Build from baseline/compare/metric (if you prefer)
# import re
# def _sanitize_for_path(name: str) -> str:
#     return re.sub(r"[^\w\-]", "_", str(name).strip()).strip("_") or "model"
# BASELINE_NAME = "SAM-H (New)"
# COMPARE_NAME = "FiLM Rosie Weights"
# METRIC = "bPQ"
# PLOTS_DIR = Path("./comparison_outputs/disagreement_plots") / f"{_sanitize_for_path(BASELINE_NAME)}_vs_{_sanitize_for_path(COMPARE_NAME)}_{METRIC}"

if not PLOTS_DIR.exists():
    raise FileNotFoundError(f"Plots folder not found: {PLOTS_DIR}")

In [ ]:
# ----------------------------
# 2) Dropdown to browse images
# ----------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPyImage

manifest_path = PLOTS_DIR / "manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    options = [(f"#{m['rank']:03d} {m['image_id']} (Δ={m['delta']:.3f})", str(PLOTS_DIR / m["file"])) for m in manifest]
else:
    # Fallback: list PNGs (exclude _legend.png)
    pngs = sorted(p for p in PLOTS_DIR.glob("rank*.png") if "_legend" not in p.name)
    options = [(p.name, str(p)) for p in pngs]

dropdown = widgets.Dropdown(
    options=options,
    value=options[0][1] if options else None,
    description="Image:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="600px"),
)
output = widgets.Output()

def on_select(change):
    with output:
        clear_output(wait=True)
        if change["new"]:
            display(IPyImage(filename=change["new"], width=800))

try:
    dropdown.unobserve(on_select, names="value")
except Exception:
    pass
dropdown.observe(on_select, names="value")
display(widgets.VBox([dropdown, output]))
if options:
    on_select({"new": options[0][1]})